# 04 · Gold — Fato Acidente | Acidentes ANTT

| | |
|---|---|
| **Origem** | Delta Table `silver_acidentes` + dims `gold_dim_*` |
| **Destino** | Delta Table `gold_fato_acidente` |
| **Execução no Pipeline** | Etapa 2 (paralelo) — requer `03_nb_gold_dims` completo |
| **Idempotente** | Sim — `overwrite` recria a tabela |

**Schema:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `id_acidente` | long | Dimensão degenerada (xxhash64) — sem relacionamento no Power BI |
| `id_data` | int | FK → `gold_dim_data` |
| `id_concessionaria` | int | FK → `gold_dim_concessionaria` |
| `id_rodovia` | int | FK → `gold_dim_rodovia` |
| `id_tipo_acidente` | int | FK → `gold_dim_tipo_acidente` |
| `km`, `hora`, `sentido` | — | Atributos degenerados |
| `tipo_de_ocorrencia`, `severidade` | — | Atributos degenerados |
| `total_veiculos`, `total_vitimas` | int | Métricas de contagem |
| `mortos` … `ilesos` | int | Métricas de vítimas por severidade |

## 1 · Imports

In [ ]:
from pyspark.sql import DataFrame, functions as F

## 2 · Parâmetros

> Célula marcada como **Parameters** — valores podem ser sobrescritos por um Data Pipeline.

In [ ]:
NOTEBOOK_NAME: str = "gold_fato_acidente"
TABELA_SILVER: str = "silver_acidentes"
PREFIXO_GOLD:  str = "gold_"
MODO_ESCRITA:  str = "overwrite"
LOG_LEVEL:     str = "INFO"

## 3 · Configuração Compartilhada

In [ ]:
%run 00_nb_config

In [ ]:
# ── Otimizações Spark ─────────────────────────────────────────────────────────
# AQE: replaneja joins e reparticiona resultados em tempo de execucao (Spark 3.3+)
spark.conf.set("spark.sql.adaptive.enabled",                    "true")
# coalescePartitions: AQE consolida particoes pequenas dinamicamente apos shuffles
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# Dimensoes <= 100 MB viram broadcast join automaticamente (evita sort-merge join)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",          str(100 * 1024 * 1024))
log.info("Spark: AQE=on | coalescePartitions=on | broadcast_threshold=100MB")

## 4 · Leitura Silver e Dimensões

In [ ]:
# Silver com surrogate key deterministico (mesmo algoritmo nos 3 notebooks de fato)
df = (
    spark.table(TABELA_SILVER)
    .withColumn(
        "id_acidente",
        F.xxhash64(
            F.col("data"),
            F.col("concessionaria"),
            F.col("km"),
            F.col("hora"),
            F.col("tipo_de_acidente"),
        ),
    )
)
log.info("Silver lida: %d registros", df.count())

# Dims ja persistidas por 03_nb_gold_dims
dim_data          = spark.table(f"{PREFIXO_GOLD}dim_data")
dim_concessionaria = spark.table(f"{PREFIXO_GOLD}dim_concessionaria")
dim_rodovia       = spark.table(f"{PREFIXO_GOLD}dim_rodovia")
dim_tipo_acidente = spark.table(f"{PREFIXO_GOLD}dim_tipo_acidente")
log.info("Dimensões carregadas.")

## 5 · Construção do Fato

In [ ]:
fato_acidente = (
    df
    # F.broadcast(): hint explicito — garante broadcast join mesmo se AQE nao inferir
    # Dimensoes sao pequenas (<1 MB): broadcast elimina o shuffle do sort-merge join
    .join(F.broadcast(dim_data.select("data", "id_data")),
          on="data", how="left")
    .join(F.broadcast(dim_concessionaria.select("concessionaria", "id_concessionaria")),
          on="concessionaria", how="left")
    .join(F.broadcast(dim_rodovia.select("rodovia", "uf", "id_rodovia")),
          on=["rodovia", "uf"], how="left")
    .join(F.broadcast(dim_tipo_acidente.select("tipo_de_acidente", "id_tipo_acidente")),
          on="tipo_de_acidente", how="left")
    # ano: coluna de particao (pruning em queries filtradas por ano no Power BI)
    .withColumn("ano", (F.col("id_data") / 10000).cast("int"))
    .select(
        "id_acidente",
        "id_data",
        "ano",
        "id_concessionaria",
        "id_rodovia",
        "id_tipo_acidente",
        "km",
        "hora",
        "sentido",
        "tipo_de_ocorrencia",
        "severidade",
        "total_veiculos",
        "total_vitimas",
        "mortos",
        "gravemente_feridos",
        "moderadamente_feridos",
        "levemente_feridos",
        "ilesos",
    )
)

log.info("fato_acidente construido: %d linhas", fato_acidente.count())

## 6 · Persistência

In [ ]:
nome = f"{PREFIXO_GOLD}fato_acidente"
(
    fato_acidente.write
    .format("delta")
    .mode(MODO_ESCRITA)
    .option("overwriteSchema", "true")
    .partitionBy("ano")          # pruning de particao em queries filtradas por ano
    .saveAsTable(nome)
)
total = spark.sql(f"SELECT COUNT(*) FROM {nome}").collect()[0][0]
log.info("%s salva: %d registros", nome, total)

# OPTIMIZE compacta arquivos pequenos gerados pelo Spark
# ZORDER ordena fisicamente os dados pelas colunas mais usadas em filtros do Power BI
spark.sql(f"OPTIMIZE {nome} ZORDER BY (id_data, id_concessionaria, id_rodovia)")
log.info("OPTIMIZE ZORDER aplicado em %s", nome)

## 7 · Relatório

In [ ]:
log.info("=== Schema: %sfato_acidente ===", PREFIXO_GOLD)
spark.sql(f"DESCRIBE TABLE {PREFIXO_GOLD}fato_acidente").show(30, truncate=False)

log.info("=== Distribuicao por severidade ===")
spark.sql(f"""
    SELECT severidade, COUNT(*) AS acidentes
    FROM {PREFIXO_GOLD}fato_acidente
    GROUP BY severidade
    ORDER BY acidentes DESC
""").show(truncate=False)

notebookutils.notebook.exit(f"OK: {PREFIXO_GOLD}fato_acidente criada com {total} registros.")